# 10 — Dequantized Matrix Multiplication (W4A16 INT4 GEMM)

## Background

LLM inference is primarily **memory-bandwidth bound**: each autoregressive decoding step requires loading the full model weights from HBM once. For a 70B-parameter model:

| Precision | Weight size | Bandwidth per token |
|-----------|------------|--------------------|
| FP16 | 140 GB | 140 GB |
| INT4 | 35 GB  | 35 GB (4× savings) |

**W4A16 quantisation** (weight-only INT4 + FP16 activations) is the dominant LLM quantisation approach (GPTQ, AWQ, llm.int4()).

## INT4 Packing Format

Each `uint8` byte stores **2 INT4 values**:
```
low  nibble: B_dequant[k, 2j]   = float16(B_packed[k,j] & 0x0F) - 8
high nibble: B_dequant[k, 2j+1] = float16((B_packed[k,j] >> 4) & 0x0F) - 8
```

**Fused kernel strategy**: dequantise INT4 on-the-fly inside the K loop — no explicit FP16 B matrix is ever materialised (saves 50% memory and bandwidth).

## Problem Definition

$$C[i, j] = \sum_{k=0}^{K-1} A[i, k] \times \text{Dequant}(B_{\text{packed}},\ k,\ j)$$

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch Reference

In [ ]:
M, K, N = 512, 512, 512
A        = torch.randn(M, K, dtype=torch.float16, device="cuda")
B_packed = torch.randint(0, 256, (K, N // 2), dtype=torch.uint8, device="cuda")

def ref_dequant_mm(A, B_packed):
    """Fully dequantise B to FP16 first, then matmul (reference; memory-inefficient)."""
    K_, N_half = B_packed.shape;  N_ = N_half * 2
    B = torch.zeros(K_, N_, dtype=torch.float16, device=A.device)
    B[:, 0::2] = (B_packed & 0x0F).float().to(torch.float16) - 8.0
    B[:, 1::2] = ((B_packed >> 4) & 0x0F).float().to(torch.float16) - 8.0
    return torch.matmul(A, B)

C_ref = ref_dequant_mm(A, B_packed)
print(f"A: {A.shape} (FP16)")
print(f"B_packed: {B_packed.shape} (uint8, INT4×2) → represents [{K}, {N}]")
print(f"C: {C_ref.shape} (FP16)")

## Triton Implementation

Inside the K loop, load `B_packed` tile and unpack INT4 via bitwise ops:
- `packed_cols = cols_n // 2` maps N column indices to packed column indices
- `shifts = (cols_n % 2) * 4` selects the low (0) or high (4) nibble
- `tl.dot(a, b)` triggers Tensor Core MMA after dequantisation

In [ ]:
@triton.jit
def _dequant_mm_kernel(A_ptr, Bp_ptr, C_ptr, M, N, K, N_half,
                       BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid_m = tl.program_id(0);  pid_n = tl.program_id(1)
    rows_m = pid_m * BM + tl.arange(0, BM)
    cols_n = pid_n * BN + tl.arange(0, BN)
    acc = tl.zeros([BM, BN], dtype=tl.float32)

    packed_cols = cols_n // 2        # [BN]  0,0,1,1,...
    shifts      = (cols_n % 2) * 4  # [BN]  0,4,0,4,...

    for k_start in range(0, K, BK):
        k_range = k_start + tl.arange(0, BK)
        mask_a  = (rows_m[:, None] < M) & (k_range[None, :] < K)
        a = tl.load(A_ptr + rows_m[:, None] * K + k_range[None, :],
                    mask=mask_a, other=0.0)
        mask_b = (k_range[:, None] < K) & (packed_cols[None, :] < N_half)
        bp = tl.load(Bp_ptr + k_range[:, None] * N_half + packed_cols[None, :],
                     mask=mask_b, other=0).to(tl.int32)
        b = ((bp >> shifts[None, :]) & 0xF).to(tl.float16) - 8.0
        acc = acc + tl.dot(a, b, allow_tf32=False)

    mask_c = (rows_m[:, None] < M) & (cols_n[None, :] < N)
    tl.store(C_ptr + rows_m[:, None] * N + cols_n[None, :],
             acc.to(tl.float16), mask=mask_c)

BM_t, BN_t, BK_t = 64, 64, 32

def triton_dequant_mm(A, B_packed):
    M_, K_ = A.shape;  _, N_half = B_packed.shape;  N_ = N_half * 2
    C = torch.empty(M_, N_, dtype=torch.float16, device=A.device)
    grid = (triton.cdiv(M_, BM_t), triton.cdiv(N_, BN_t))
    _dequant_mm_kernel[grid](A, B_packed, C, M_, N_, K_, N_half,
                             BM=BM_t, BN=BN_t, BK=BK_t)
    return C

C_tri = triton_dequant_mm(A, B_packed)
ok = torch.allclose(C_ref, C_tri, atol=0.5, rtol=0.1)
print("Triton correctness:", "✅ PASSED" if ok else
      f"❌ FAILED  max_diff={torch.max(torch.abs(C_ref - C_tri)).item():.4f}")

## TileLang Implementation

Inside the K loop, a nested `T.Serial(BLOCK_N // 2)` loop unpacks each packed byte: extract low nibble (even column) and high nibble (odd column), accumulate directly into the float32 accumulator.

In [ ]:
# Fixed for gfx1100: load B_packed into shared memory, dequantize with two
# separate T.Parallel loops (one per nibble) to satisfy layout inference, then
# call T.gemm for Tensor-Core-accelerated matrix multiplication.
BM, BN, BK = 128, 128, 64

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_dequant_mm(A, B_packed, BLOCK_M: int, BLOCK_N: int, BLOCK_K: int):
    M_, K_, N_ = T.const("M, K, N")
    dtype, acc_dtype = T.float16, T.float32
    A:        T.Tensor((M_, K_),      dtype)
    B_packed: T.Tensor((K_, N_ // 2), T.uint8)
    C = T.empty((M_, N_), dtype)
    with T.Kernel(T.ceildiv(N_, BLOCK_N), T.ceildiv(M_, BLOCK_M), threads=128) as (pid_n, pid_m):
        A_s  = T.alloc_shared((BLOCK_M, BLOCK_K), dtype)
        Bp_s = T.alloc_shared((BLOCK_K, BLOCK_N // 2), T.uint8)
        B_s  = T.alloc_shared((BLOCK_K, BLOCK_N), dtype)
        C_l  = T.alloc_fragment((BLOCK_M, BLOCK_N), acc_dtype)
        T.clear(C_l)
        for k in T.Serial(K_ // BLOCK_K):
            T.copy(A[pid_m * BLOCK_M, k * BLOCK_K], A_s)
            T.copy(B_packed[k * BLOCK_K, pid_n * BLOCK_N // 2], Bp_s)
            # Dequantize low nibbles (even columns) — separate loop for layout inference
            for j, jj in T.Parallel(BLOCK_K, BLOCK_N // 2):
                B_s[j, 2 * jj] = (Bp_s[j, jj] & T.uint8(0x0F)).astype(dtype) - dtype(8)
            # Dequantize high nibbles (odd columns)
            for j, jj in T.Parallel(BLOCK_K, BLOCK_N // 2):
                B_s[j, 2 * jj + 1] = ((Bp_s[j, jj] >> T.uint8(4)) & T.uint8(0x0F)).astype(dtype) - dtype(8)
            T.gemm(A_s, B_s, C_l)
        T.copy(C_l, C[pid_m * BLOCK_M, pid_n * BLOCK_N])
    return C

k = tl_dequant_mm.compile(M=M, K=K, N=N, BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK)
C_tl = k(A.clone(), B_packed.clone())
ok = torch.allclose(C_ref, C_tl, atol=0.5, rtol=0.1)
print("TileLang correctness:", "\u2705 PASSED" if ok else
      f"\u274c FAILED  max_diff={torch.max(torch.abs(C_ref - C_tl)).item():.4f}")


## Performance Comparison

Benchmark with larger matrices (M=K=N=4096) to clearly show the advantage of fused dequantisation over the explicit dequant → matmul approach.

In [ ]:
import matplotlib.pyplot as plt

M_b, K_b, N_b = 4096, 4096, 4096
A_b   = torch.randn(M_b, K_b, dtype=torch.float16, device="cuda")
Bp_b  = torch.randint(0, 256, (K_b, N_b // 2), dtype=torch.uint8, device="cuda")

k_large = tl_dequant_mm.compile(M=M_b, K=K_b, N=N_b, BLOCK_M=BM, BLOCK_N=BN, BLOCK_K=BK)

WARMUP, REPEAT = 20, 100

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

def pytorch_dequant_mm(A, Bp):
    B = torch.zeros(K_b, N_b, dtype=torch.float16, device=A.device)
    B[:, 0::2] = (Bp & 0x0F).float().to(torch.float16) - 8.0
    B[:, 1::2] = ((Bp >> 4) & 0x0F).float().to(torch.float16) - 8.0
    return torch.matmul(A, B)

results = {
    "PyTorch\n(dequant→mm)": bench(pytorch_dequant_mm, [A_b, Bp_b]),
    "Triton\n(fused)": bench(triton_dequant_mm, [A_b, Bp_b]),
    "TileLang\n(fused)": bench(k_large, [A_b, Bp_b]),
}

flops = 2 * M_b * N_b * K_b
for name, ms in results.items():
    print(f"  {name.replace(chr(10),' '):25s}: {ms:.4f} ms  ({flops / ms / 1e9:.1f} TFLOPS)")

print(f"\nWeight memory: FP16 = {K_b*N_b*2/1e6:.0f} MB  →  INT4 = {K_b*N_b//2/1e6:.0f} MB  (4× savings)")

colors = ["#4C72B0", "#55A868", "#C44E52"]
labels = list(results.keys())
ms_vals = list(results.values())
tf_vals = [flops / ms / 1e9 for ms in ms_vals]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
bars = ax.bar(labels, ms_vals, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, ms_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(ms_vals)*0.01,
            f"{val:.2f} ms", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Latency (ms)")
ax.set_title(f"INT4 GEMM Latency\n(M=K=N={M_b}, FP16×INT4)")
ax.set_ylim(0, max(ms_vals)*1.25); ax.spines[["top","right"]].set_visible(False)

ax = axes[1]
bars = ax.bar(labels, tf_vals, color=colors, width=0.45, edgecolor="white", linewidth=0.8)
for bar, val in zip(bars, tf_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(tf_vals)*0.01,
            f"{val:.1f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Equivalent Throughput (TFLOPS)")
ax.set_title(f"INT4 GEMM Throughput\n(M=K=N={M_b}, FP16×INT4)")
ax.set_ylim(0, max(tf_vals)*1.25); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()